# ArSL Custom Words — Subset Training

Train a **smaller vocabulary** from the full KArSL-502 cache built by `ArSL_Word_Training_v2.ipynb`.

**Prerequisite:** `arsl_word_sequences_v2_full.npz` must exist (run Cell 6 in the main notebook first).

**Word list:** edit `KARSL-502_BasicWords.csv` — column `class_id` is the KArSL folder number (SignID + 1).

### Pipeline
1. Config & GPU
2. Load labels
3. Filter full NPZ → custom subset NPZ
4. Data exploration
5. Preprocessing & splits
6. Train model (`arsl_custom_*` outputs)
7. Evaluation dashboard

### Outputs
| File | Description |
|------|-------------|
| `arsl_custom_subset.npz` | Filtered sequences for your word list |
| `arsl_custom_best.h5` | Best checkpoint |
| `arsl_custom_classes.csv` | Model index ↔ KArSL class ID ↔ names |
| `arsl_custom_scaler.npz` | StandardScaler for inference |


In [ ]:
# Cell 1: Imports
import os
import time
import warnings
from pathlib import Path

os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Bidirectional, Dense, Dropout,
    BatchNormalization, TimeDistributed
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import mixed_precision

warnings.filterwarnings('ignore')
print(f'TensorFlow {tf.__version__} | NumPy {np.__version__}')

In [ ]:
# Cell 2: GPU
gpus = tf.config.list_physical_devices('GPU')
USE_GPU = False
DEVICE = '/CPU:0'

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        USE_GPU = True
        DEVICE = '/GPU:0'
        print(f'GPU: {gpus[0].name}')
    except RuntimeError as e:
        print(f'GPU config failed: {e}')
else:
    print('No GPU — using CPU')

mixed_precision.set_global_policy('float32')
print(f'Device: {DEVICE}')

In [ ]:
# Cell 3: Configuration
PROJECT_ROOT = Path(r'M:/Term 10/Grad')
OUTPUT_DIR = PROJECT_ROOT / 'SLR Main' / 'Words' / 'ArSL Word (Arabic)'
LABELS_FILE = OUTPUT_DIR / 'KARSL-502_Labels.txt'

# Input: full cache from main notebook
FULL_NPZ_PATH = OUTPUT_DIR / 'arsl_word_sequences_v2_full.npz'

# Custom vocabulary — edit this CSV to add/remove words
CUSTOM_WORDS_CSV = OUTPUT_DIR / 'KARSL-502_BasicWords.csv'
SUBSET_NPZ_PATH = OUTPUT_DIR / 'arsl_custom_subset.npz'
OUTPUT_PREFIX = 'arsl_custom'

# Feature layout (must match main notebook)
POSE_FEATURES = 33 * 4
HAND_FEATURES = 21 * 3
NUM_FEATURES = POSE_FEATURES + HAND_FEATURES * 2  # 258
SEQUENCE_LENGTH = 48

# Hyperparameters — smaller model OK for ~45 classes
BATCH_SIZE = 64
EPOCHS = 150
LEARNING_RATE = 5e-4
LSTM_UNITS_1 = 128
LSTM_UNITS_2 = 96
LSTM_UNITS_3 = 64
SPATIAL_ENC_1 = 192
SPATIAL_ENC_2 = 128
DENSE_UNITS = 256
DROPOUT_RATE = 0.35
LABEL_SMOOTH = 0.05
GRAD_CLIP_NORM = 1.0
TEST_SIZE = 0.4

print('=' * 55)
print('CUSTOM WORDS TRAINING — CONFIG')
print('=' * 55)
print(f'  Full NPZ      : {FULL_NPZ_PATH}  [{"OK" if FULL_NPZ_PATH.exists() else "MISSING — run main notebook Cell 6"}]')
print(f'  Word list CSV : {CUSTOM_WORDS_CSV}  [{"OK" if CUSTOM_WORDS_CSV.exists() else "MISSING"}]')
print(f'  Subset cache  : {SUBSET_NPZ_PATH}')
print(f'  Output prefix : {OUTPUT_PREFIX}_*')
print(f'  Batch / epochs: {BATCH_SIZE} / {EPOCHS}')

In [ ]:
# Cell 4: Load KArSL labels
id_to_english = {}
id_to_arabic = {}

if LABELS_FILE.exists():
    with open(LABELS_FILE, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('SignID'):
                continue
            parts = line.split('\t')
            if len(parts) >= 3:
                try:
                    sid = int(parts[0])
                    ar = parts[1].strip()
                    en = parts[2].strip()
                    mapped_id = sid + 1
                    id_to_english[mapped_id] = en if en and en not in ('?', '??', '') else str(mapped_id)
                    id_to_arabic[mapped_id] = ar if ar and ar not in ('?', '??', '') else en
                except Exception:
                    continue
    print(f'Labels loaded: {len(id_to_english)} entries')
else:
    print('Labels file not found — numeric IDs will be used')

In [ ]:
# Cell 5: Build custom subset from full NPZ
print('=' * 55)
print('BUILD CUSTOM WORD SUBSET')
print('=' * 55)

if not FULL_NPZ_PATH.exists():
    raise FileNotFoundError(
        f'Full NPZ not found: {FULL_NPZ_PATH}\n'
        f'Finish extraction in ArSL_Word_Training_v2.ipynb (Cell 6) first.'
    )
if not CUSTOM_WORDS_CSV.exists():
    raise FileNotFoundError(f'Word list not found: {CUSTOM_WORDS_CSV}')

words_df = pd.read_csv(CUSTOM_WORDS_CSV)
if 'class_id' not in words_df.columns:
    raise ValueError('CSV must have a class_id column')

target_ids = sorted(words_df['class_id'].astype(int).unique().tolist())
print(f'Custom vocabulary: {len(target_ids)} words')

with np.load(str(FULL_NPZ_PATH), mmap_mode='r') as _d:
    full_shape = _d['X'].shape
    print(f'Full cache shape: {full_shape}')

_d = np.load(str(FULL_NPZ_PATH))
X_full, y_full = _d['X'], _d['y']
del _d

available = set(int(v) for v in np.unique(y_full))
missing = [cid for cid in target_ids if cid not in available]

mask = np.isin(y_full, target_ids)
X, y = X_full[mask], y_full[mask]
del X_full, y_full

if len(y) == 0:
    raise RuntimeError('No samples matched your word list — check class_id values in the CSV.')

np.savez_compressed(str(SUBSET_NPZ_PATH), X=X, y=y)

for cid in target_ids:
    id_to_english.setdefault(cid, str(cid))
    id_to_arabic.setdefault(cid, str(cid))

print(f'\nSubset saved : {SUBSET_NPZ_PATH}')
print(f'  Samples    : {len(y):,}')
print(f'  Classes    : {len(np.unique(y))} / {len(target_ids)} requested')
if missing:
    print(f'  Missing IDs: {missing}')
else:
    print('  All requested words found in full NPZ')

preview_cols = ['class_id', 'english'] if 'english' in words_df.columns else ['class_id']
counts_df = pd.DataFrame({
    'class_id': target_ids,
    'samples': [int((y == cid).sum()) for cid in target_ids],
})
preview = words_df[preview_cols].drop_duplicates('class_id').merge(counts_df, on='class_id', how='left')
print('\nVocabulary preview:')
print(preview.to_string(index=False))

In [ ]:
# Cell 6: Data exploration
unique_ids, counts = np.unique(y, return_counts=True)
names = [id_to_english.get(int(uid), str(uid)) for uid in unique_ids]

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(names, counts, color='steelblue', edgecolor='black', linewidth=0.3)
ax.set_xticklabels(names, rotation=90, fontsize=8)
ax.set_title(f'Custom subset — {len(unique_ids)} classes, {len(y)} samples')
ax.set_ylabel('Samples')
plt.tight_layout()
plt.show()

print(f'Min/max per class: {counts.min()} / {counts.max()}')

In [ ]:
# Cell 7: Preprocessing & splits
print('Preprocessing ...')

orig_shape = X.shape
X_flat = X.reshape(-1, NUM_FEATURES)
scaler = StandardScaler()
X_flat = scaler.fit_transform(X_flat)
X = X_flat.reshape(orig_shape).astype(np.float32)

np.savez_compressed(
    str(OUTPUT_DIR / f'{OUTPUT_PREFIX}_scaler.npz'),
    mean=scaler.mean_.astype(np.float32),
    scale=scaler.scale_.astype(np.float32),
)

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
num_classes = len(encoder.classes_)
y_onehot = to_categorical(y_encoded, num_classes=num_classes)

class_df = pd.DataFrame({
    'model_class_index': range(num_classes),
    'karsl_class_id': encoder.classes_.tolist(),
    'english': [id_to_english.get(int(c), str(c)) for c in encoder.classes_],
    'arabic': [id_to_arabic.get(int(c), str(c)) for c in encoder.classes_],
})
class_df.to_csv(OUTPUT_DIR / f'{OUTPUT_PREFIX}_classes.csv', index=False)
print(f'Class map saved ({num_classes} classes)')
print(class_df.head(10).to_string())

try:
    X_train, X_tmp, y_train, y_tmp = train_test_split(
        X, y_onehot, test_size=TEST_SIZE, random_state=42, stratify=y_encoded
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=np.argmax(y_tmp, 1)
    )
except ValueError:
    print('Stratified split failed — using random split')
    X_train, X_tmp, y_train, y_tmp = train_test_split(X, y_onehot, test_size=TEST_SIZE, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42)

train_ints = np.argmax(y_train, axis=1)
cw_array = compute_class_weight('balanced', classes=np.arange(num_classes), y=train_ints)
cw_array = np.clip(cw_array, 0.5, 10.0)
class_weights = dict(enumerate(cw_array))

print(f'Train {X_train.shape[0]} | Val {X_val.shape[0]} | Test {X_test.shape[0]}')

In [ ]:
# Cell 8: Build & train
tf.keras.backend.clear_session()

POSE_F = tf.constant(POSE_FEATURES, dtype=tf.int32)
HAND_F = tf.constant(HAND_FEATURES, dtype=tf.int32)

def augment_sequence(x, y_label):
    x = x + tf.random.normal(tf.shape(x), mean=0.0, stddev=0.005)
    x = tf.roll(x, shift=tf.random.uniform([], -3, 4, dtype=tf.int32), axis=0)
    mask = tf.cast(tf.random.uniform([SEQUENCE_LENGTH, 1]) > 0.1, tf.float32)
    x = x * mask
    if tf.random.uniform([]) > 0.5:
        pose = x[:, :POSE_F]
        lh = x[:, POSE_F:POSE_F + HAND_F]
        rh = x[:, POSE_F + HAND_F:]
        x = tf.concat([pose, rh, lh], axis=-1)
    return x, y_label

AUTOTUNE = tf.data.AUTOTUNE
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(min(len(X_train), 5000), seed=42)
    .map(augment_sequence, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(AUTOTUNE)

inputs = Input(shape=(SEQUENCE_LENGTH, NUM_FEATURES), name='landmark_input')
x = TimeDistributed(Dense(SPATIAL_ENC_1, activation='relu'), name='td_enc_1')(inputs)
x = TimeDistributed(BatchNormalization(), name='td_bn_1')(x)
x = TimeDistributed(Dropout(0.2), name='td_drop_1')(x)
x = TimeDistributed(Dense(SPATIAL_ENC_2, activation='relu'), name='td_enc_2')(x)
x = TimeDistributed(BatchNormalization(), name='td_bn_2')(x)
x = Bidirectional(LSTM(LSTM_UNITS_1, return_sequences=True), name='bilstm_1')(x)
x = BatchNormalization(name='bn_1')(x)
x = tf.keras.layers.SpatialDropout1D(DROPOUT_RATE, name='sdrop_1')(x)
x = Bidirectional(LSTM(LSTM_UNITS_2, return_sequences=True), name='bilstm_2')(x)
x = BatchNormalization(name='bn_2')(x)
x = tf.keras.layers.SpatialDropout1D(DROPOUT_RATE, name='sdrop_2')(x)
x = LSTM(LSTM_UNITS_3, return_sequences=False, name='lstm_final')(x)
x = BatchNormalization(name='bn_lstm_final')(x)
x = Dropout(DROPOUT_RATE, name='drop_lstm')(x)
x = Dense(DENSE_UNITS, activation='relu', name='dense_1')(x)
x = BatchNormalization(name='bn_dense_1')(x)
x = Dropout(DROPOUT_RATE, name='drop_1')(x)
x = Dense(DENSE_UNITS // 2, activation='relu', name='dense_2')(x)
x = Dropout(DROPOUT_RATE * 0.5, name='drop_2')(x)
outputs = Dense(num_classes, activation='softmax', dtype='float32', name='output')(x)

model = Model(inputs, outputs, name='ArSL_CustomWords')
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=GRAD_CLIP_NORM)
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH)
model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')],
)
model.summary()

MODEL_BEST = str(OUTPUT_DIR / f'{OUTPUT_PREFIX}_best.h5')
MODEL_FINAL = str(OUTPUT_DIR / f'{OUTPUT_PREFIX}_final.h5')
callbacks = [
    ModelCheckpoint(MODEL_BEST, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.TerminateOnNaN(),
]

print(f'Training on {DEVICE} ...')
t0 = time.time()
with tf.device(DEVICE):
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1,
    )
print(f'Done in {(time.time()-t0)/60:.1f} min | best val_acc {max(history.history["val_accuracy"]):.4f}')
model.save(MODEL_FINAL)
print(f'Saved: {MODEL_BEST}')

In [ ]:
# Cell 9: Evaluation
best_model = tf.keras.models.load_model(MODEL_BEST)
with tf.device(DEVICE):
    proba = best_model.predict(test_ds, verbose=0)

y_pred = np.argmax(proba, axis=1)
y_true = np.argmax(y_test, axis=1)
top1 = (y_pred == y_true).mean()
top5 = sum(1 for i in range(len(y_true)) if y_true[i] in np.argsort(proba[i])[-5:]) / len(y_true)

word_labels = [id_to_english.get(int(encoder.classes_[i]), str(encoder.classes_[i])) for i in range(num_classes)]
print(f'Test Top-1: {top1*100:.2f}% | Top-5: {top5*100:.2f}% | n={len(y_true)}')
print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=word_labels, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / f'{OUTPUT_PREFIX}_training_curves.png'), dpi=150)
plt.show()

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(max(8, num_classes * 0.35), max(6, num_classes * 0.3)))
sns.heatmap(cm, xticklabels=word_labels, yticklabels=word_labels, cmap='Blues', fmt='d')
plt.title('Confusion matrix — custom words')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / f'{OUTPUT_PREFIX}_confusion_matrix.png'), dpi=150)
plt.show()

### Troubleshooting

| Issue | Fix |
|-------|-----|
| Full NPZ missing | Finish `ArSL_Word_Training_v2.ipynb` Cell 6 first |
| Missing class IDs | That word was not in the full extraction yet — wait or re-run main Cell 6 |
| Change vocabulary | Edit `KARSL-502_BasicWords.csv`, re-run from Cell 5 |
| OOM on MX150 | Set `BATCH_SIZE = 32` in Cell 3 |
